## Import libraries and extract data

In [1]:
import numpy as np
import tensorflow as tf

from sklearn import preprocessing

raw_data = np.loadtxt('Audiobooks_data.csv', delimiter=',')

unscaled_inputs = raw_data[:, 1:-1]
targets = raw_data[:, -1]

#### Balance the dataset

In [2]:
one_targets = int(np.sum(targets))
zero_targets = 0
indices_to_remove = []

for i in range(targets.shape[0]):
    if targets[i] == 0:
        if zero_targets < one_targets:
            zero_targets += 1
        else:
            indices_to_remove.append(i)
print('Number of indices to remove: ', len(indices_to_remove))

unscaled_inputs_balanced = np.delete(unscaled_inputs, indices_to_remove, axis=0)
targets_balanced = np.delete(targets, indices_to_remove, axis=0)

print('Targets after balancing: {0:2f} // Inputs after balancing: {1:2f} '.format(targets_balanced.shape[0], unscaled_inputs_balanced.shape[0]))

Number of indices to remove:  9610
Targets after balancing: 4474.000000 // Inputs after balancing: 4474.000000 


### Standardize the inputs

In [3]:
scaled_inputs = preprocessing.scale(unscaled_inputs_balanced)

### Shuffle the data

In [4]:
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_balanced[shuffled_indices]

### Split the dataset into train, validation and test sets

In [5]:
sample_count = shuffled_inputs.shape[0]

train_samples_count = int(0.8 * sample_count)
validation_samples_count = int(0.1 * sample_count)
test_samples_count = sample_count - train_samples_count - validation_samples_count

train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count: train_samples_count + validation_samples_count]
validation_targets = shuffled_targets[train_samples_count: train_samples_count + validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count + validation_samples_count:]
test_targets = shuffled_targets[train_samples_count + validation_samples_count:]

print('Training set: {0} samples // Validation set: {1} samples // Test set: {2} samples'.format(train_inputs.shape[0], validation_inputs.shape[0], test_inputs.shape[0]))


Training set: 3579 samples // Validation set: 447 samples // Test set: 448 samples


### Save the datasets into .npz files

In [6]:
np.savez('Audiobooks_train.npz', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_validation.npz', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_test.npz', inputs=test_inputs, targets=test_targets)